## 1. Load earlier training data only


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score

data_path = Path("data/bank-additional-full.csv")

df = pd.read_csv(data_path, sep=";")

X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})

train_end = int(len(df) * 0.60)

X_train_check = X.iloc[:train_end].copy()
y_train_check = y.iloc[:train_end].copy()

check_cv = TimeSeriesSplit(n_splits=3)

assert X_train_check.shape[0] == 24712
assert y_train_check.isin([0, 1]).all()

print("Training observations:", len(X_train_check))
print("Training positive rate:", f"{y_train_check.mean():.2%}")

Training observations: 24712
Training positive rate: 4.81%


## 2. All dummy baselines


In [2]:
from sklearn.dummy import DummyClassifier

dummy_strategies = {
    "prior": DummyClassifier(strategy="prior"),
    "most_frequent": DummyClassifier(strategy="most_frequent"),
    "stratified": DummyClassifier(
        strategy="stratified", random_state=42
    ),
    "uniform": DummyClassifier(
        strategy="uniform", random_state=42
    ),
    "always_negative": DummyClassifier(
        strategy="constant", constant=0
    ),
    "always_positive": DummyClassifier(
        strategy="constant", constant=1
    )
}

dummy_rows = []

for name, dummy in dummy_strategies.items():

    fold_scores = []

    for train_idx, val_idx in check_cv.split(X_train_check):

        X_fold_train = X_train_check.iloc[train_idx]
        y_fold_train = y_train_check.iloc[train_idx]

        X_fold_val = X_train_check.iloc[val_idx]
        y_fold_val = y_train_check.iloc[val_idx]

        dummy.fit(X_fold_train, y_fold_train)

        probabilities = dummy.predict_proba(
            X_fold_val
        )[:, list(dummy.classes_).index(1)]

        score = average_precision_score(
            y_fold_val,
            probabilities
        )

        fold_scores.append(score)

    dummy_rows.append({
        "Strategy": name,
        "Fold 1 AP": fold_scores[0],
        "Fold 2 AP": fold_scores[1],
        "Fold 3 AP": fold_scores[2],
        "Mean AP": np.mean(fold_scores),
        "Fold SD": np.std(fold_scores, ddof=0)
    })

dummy_results = pd.DataFrame(dummy_rows)

display(
    dummy_results.sort_values(
        "Mean AP",
        ascending=False
    ).round(5)
)

,Strategy,Fold 1 AP,Fold 2 AP,Fold 3 AP,Mean AP,Fold SD
2,stratified,0.04032,0.06689,0.05814,0.05512,0.01106
0,prior,0.04047,0.06394,0.05827,0.05422,0.01000
1,most_frequent,0.04047,0.06394,0.05827,0.05422,0.01000
3,uniform,0.04047,0.06394,0.05827,0.05422,0.01000
4,always_negative,0.04047,0.06394,0.05827,0.05422,0.01000
5,always_positive,0.04047,0.06394,0.05827,0.05422,0.01000


## 3. Single-feature screening


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


def one_feature_pipeline(feature):

    is_numeric = pd.api.types.is_numeric_dtype(
        X_train_check[feature]
    )

    if is_numeric:
        transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

    else:
        transformer = Pipeline([
            ("imputer", SimpleImputer(
                missing_values="unknown",
                strategy="constant",
                fill_value="Missing"
            )),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore"
            ))
        ])

    preprocessor = ColumnTransformer([
        ("chosen_feature", transformer, [feature])
    ])

    return Pipeline([
        ("pre", preprocessor),
        ("model", LogisticRegression(
            C=1.0,
            max_iter=3000,
            random_state=42
        ))
    ])


single_feature_rows = []

for feature in X_train_check.columns:

    pipeline = one_feature_pipeline(feature)

    scores = cross_val_score(
        pipeline,
        X_train_check[[feature]],
        y_train_check,
        scoring="average_precision",
        cv=check_cv,
        n_jobs=1
    )

    single_feature_rows.append({
        "Feature": feature,
        "Mean AP": scores.mean(),
        "Fold SD": scores.std(),
        "Fold 1": scores[0],
        "Fold 2": scores[1],
        "Fold 3": scores[2]
    })

single_feature_results = pd.DataFrame(
    single_feature_rows
).sort_values(
    "Mean AP",
    ascending=False
)

display(single_feature_results.round(5))

,Feature,Mean AP,Fold SD,Fold 1,Fold 2,Fold 3
0,age,0.05933,0.01093,0.04387,0.06715,0.06697
17,euribor3m,0.05822,0.01385,0.03977,0.06175,0.07315
3,education,0.05667,0.00979,0.04323,0.06625,0.06054
9,day_of_week,0.05643,0.01111,0.04189,0.06884,0.05858
1,job,0.05552,0.01007,0.04149,0.06466,0.06041
4,default,0.05550,0.01084,0.04045,0.06556,0.06049
2,marital,0.05502,0.00954,0.04184,0.06409,0.05914
5,housing,0.05476,0.01129,0.03969,0.06687,0.05773
16,cons.conf.idx,0.05471,0.01056,0.03993,0.06394,0.06027
10,campaign,0.05452,0.01085,0.03918,0.06248,0.06190


## 4. Age-band investigation


In [4]:
# Age bands chosen before inspecting their outcomes
age_edges = [0, 30, 40, 50, 60, 120]

age_labels = [
    "Under 30",
    "30–39",
    "40–49",
    "50–59",
    "60+"
]

age_rows = []

for fold, (_, val_idx) in enumerate(
    check_cv.split(X_train_check),
    start=1
):
    # Select the corresponding fold's age and labels
    fold_age = X_train_check.iloc[val_idx]["age"]
    fold_y = y_train_check.iloc[val_idx]

    # Assign customers to age ranges
    bands = pd.cut(
        fold_age,
        bins=age_edges,
        labels=age_labels,
        right=False
    )

    fold_table = pd.DataFrame({
        "Age band": bands,
        "Subscribed": fold_y
    })

    grouped = fold_table.groupby(
        "Age band",
        observed=True
    )["Subscribed"].agg(["count", "mean"])

    for band, values in grouped.iterrows():
        age_rows.append({
            "Fold": fold,
            "Age band": str(band),
            "Customers": int(values["count"]),
            "Subscription rate": float(values["mean"])
        })

age_by_fold = pd.DataFrame(age_rows)

display(age_by_fold.round(4))

,Fold,Age band,Customers,Subscription rate
0,1,Under 30,728,0.0440
1,1,30–39,2493,0.0421
2,1,40–49,1827,0.0378
3,1,50–59,1087,0.0386
4,1,60+,43,0.0465
5,2,Under 30,1248,0.0625
6,2,30–39,2233,0.0766
7,2,40–49,1546,0.0472
8,2,50–59,1113,0.0620
9,2,60+,38,0.1053


## 5. Define preprocessing


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_columns = X_train_check.select_dtypes(include="number").columns.to_list()
categorical_columns = X_train_check.select_dtypes(exclude="number").columns.to_list()

def build_preprocessor():

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(missing_values="unknown", strategy="most_frequent", fill_value="Missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ])

    return preprocessor

print("Numerical columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

Numerical columns: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


## 6. Define model families


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


logistic_pipeline = Pipeline([
    ("pre", build_preprocessor()),
    ("model", LogisticRegression(solver="lbfgs", max_iter=3000, random_state=42))
])


forest_pipeline = Pipeline([
    ("pre", build_preprocessor()),
    ("model", RandomForestClassifier(random_state=42, n_jobs=1))
])

print("Both model pipelines defined.")

Both model pipelines defined.


## 7. Five-window split stability


In [7]:
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score

# Fixed selected settings from our original searches
logistic_settings = {
    "model__C": 67.32248920775338,
    "model__class_weight": "balanced",
    "pre__categorical__imputer__strategy": "constant"
}

forest_settings = {
    "model__max_depth": 15,
    "model__max_features": 1.0,
    "model__min_samples_leaf": 0.028336828367886542,
    "model__n_estimators": 116,
    "pre__categorical__imputer__strategy": "most_frequent"
}

# Five chronological folds, using training data only
stability_cv = TimeSeriesSplit(n_splits=5)

stability_rows = []

for fold, (train_idx, val_idx) in enumerate(
    stability_cv.split(X_train_check),
    start=1
):
    X_past = X_train_check.iloc[train_idx]
    y_past = y_train_check.iloc[train_idx]

    X_future = X_train_check.iloc[val_idx]
    y_future = y_train_check.iloc[val_idx]

    assert train_idx.max() < val_idx.min()

    # Start with fresh copies of the model pipelines
    logistic_model = clone(logistic_pipeline)
    logistic_model.set_params(**logistic_settings)

    forest_model = clone(forest_pipeline)
    forest_model.set_params(**forest_settings)

    # Fit only on earlier observations
    logistic_model.fit(X_past, y_past)
    forest_model.fit(X_past, y_past)

    # Predict scores on the later fold
    logistic_scores = logistic_model.predict_proba(
        X_future
    )[:, 1]

    forest_scores = forest_model.predict_proba(
        X_future
    )[:, 1]

    stability_rows.append({
        "Fold": fold,
        "Train rows": len(X_past),
        "Validation rows": len(X_future),
        "Baseline AP": y_future.mean(),
        "Logistic AP": average_precision_score(
            y_future, logistic_scores
        ),
        "Forest AP": average_precision_score(
            y_future, forest_scores
        )
    })

stability_results = pd.DataFrame(stability_rows)

display(stability_results.round(5))

,Fold,Train rows,Validation rows,Baseline AP,Logistic AP,Forest AP
0,1,4122,4118,0.03521,0.04371,0.04357
1,2,8240,4118,0.04225,0.03802,0.04385
2,3,12358,4118,0.06629,0.07123,0.07860
3,4,16476,4118,0.06241,0.06288,0.07055
4,5,20594,4118,0.05464,0.04713,0.04063


## 8. Stability summary


In [8]:
original_cv_ap = {
    "Logistic": 0.0533,
    "Forest": 0.0562
}

summary_rows = []

for name, column in [
    ("Logistic", "Logistic AP"),
    ("Forest", "Forest AP")
]:
    mean_ap = stability_results[column].mean()

    std_ap = stability_results[column].std(ddof=1)

    original_ap = original_cv_ap[name]

    z_distance = (
        (original_ap - mean_ap) / std_ap
        if std_ap > 0 else np.nan
    )

    summary_rows.append({
        "Model": name,
        "Five-fold mean AP": mean_ap,
        "Five-fold SD": std_ap,
        "Original three-fold AP": original_ap,
        "Original score distance (SD)": z_distance
    })

stability_summary = pd.DataFrame(summary_rows)

display(stability_summary.round(5))

,Model,Five-fold mean AP,Five-fold SD,Original three-fold AP,Original score distance (SD)
0,Logistic,0.05260,0.01391,0.0533,0.05067
1,Forest,0.05544,0.01774,0.0562,0.04268


## 9. Blockwise label permutation


In [9]:
# Use a fixed seed so this diagnostic is reproducible
permutation_rng = np.random.default_rng(2027)

# Original chronological CV fold boundaries
original_cv = TimeSeriesSplit(n_splits=3)

# Find the four consecutive historical blocks
fold_indices = list(original_cv.split(X_train_check))

block_boundaries = [0]

for _, validation_idx in fold_indices:
    block_boundaries.append(int(validation_idx.min()))

block_boundaries.append(len(y_train_check))

# Create a copy. Never overwrite the real labels.
shuffled_y = y_train_check.copy()

for start, end in zip(
    block_boundaries[:-1],
    block_boundaries[1:]
):
    original_block = y_train_check.iloc[start:end].to_numpy()

    shuffled_block = permutation_rng.permutation(
        original_block
    )

    shuffled_y.iloc[start:end] = shuffled_block


# Verify that the overall positive count is unchanged
assert shuffled_y.sum() == y_train_check.sum()

# Verify that each chronological block preserves prevalence
for start, end in zip(
    block_boundaries[:-1],
    block_boundaries[1:]
):
    original_count = y_train_check.iloc[start:end].sum()
    shuffled_count = shuffled_y.iloc[start:end].sum()

    assert original_count == shuffled_count

print("Block boundaries:", block_boundaries)
print("Original positive count:", int(y_train_check.sum()))
print("Shuffled positive count:", int(shuffled_y.sum()))

print(
    "Share of labels changed:",
    f"{(shuffled_y.to_numpy() != y_train_check.to_numpy()).mean():.2%}"
)

Block boundaries: [0, 6178, 12356, 18534, 24712]
Original positive count: 1188
Shuffled positive count: 1188
Share of labels changed: 9.13%


## 10. Restore original search spaces


In [10]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.base import clone


# Same hyperparameter distributions as the original search
logistic_parameters = {
    "model__C": loguniform(1e-3, 1e2),

    "model__class_weight": [None, "balanced"],

    "pre__categorical__imputer__strategy": [
        "most_frequent", "constant"
    ]
}


forest_parameters = {
    "model__n_estimators": randint(60, 121),

    "model__max_depth": [5, 10, 15, None],

    "model__min_samples_leaf": loguniform(
        0.001, 0.03
    ),

    "model__max_features": [
        "sqrt", 0.5, 1.0
    ],

    "pre__categorical__imputer__strategy": [
        "most_frequent", "constant"
    ]
}

shuffled_cv = TimeSeriesSplit(n_splits=3)

print("Original search spaces reconstructed.")

Original search spaces reconstructed.


## 11. Full shuffled-label randomized searches


In [11]:
shuffled_logistic_search = RandomizedSearchCV(
    estimator=clone(logistic_pipeline),
    param_distributions=logistic_parameters,
    n_iter=20,
    scoring="average_precision",
    cv=shuffled_cv,
    n_jobs=2,
    random_state=42,
    refit=True,
    error_score="raise",
    return_train_score=True,
    verbose=1
)

shuffled_forest_search = RandomizedSearchCV(
    estimator=clone(forest_pipeline),
    param_distributions=forest_parameters,
    n_iter=20,
    scoring="average_precision",
    cv=shuffled_cv,
    n_jobs=2,
    random_state=42,
    refit=True,
    error_score="raise",
    return_train_score=True,
    verbose=1
)


# Full search: deliberately shuffled labels
shuffled_logistic_search.fit(
    X_train_check,
    shuffled_y
)

shuffled_forest_search.fit(
    X_train_check,
    shuffled_y
)

print("Both complete shuffled-label searches finished.")

Fitting 3 folds for each of 20 candidates, totalling 60 fits


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Both complete shuffled-label searches finished.


## 12. Real versus shuffled comparison


In [12]:
# Exact original best scores from your completed searches
real_logistic_ap = 0.0533
real_forest_ap = 0.0562

# Fold-specific no-skill baselines
baseline_folds = []

for _, validation_idx in shuffled_cv.split(X_train_check):
    baseline_folds.append(
        shuffled_y.iloc[validation_idx].mean()
    )

baseline_mean_ap = np.mean(baseline_folds)


comparison = pd.DataFrame([
    {
        "Model": "Logistic regression",
        "Real-label CV AP": real_logistic_ap,
        "Shuffled-label CV AP": shuffled_logistic_search.best_score_,
        "No-skill baseline AP": baseline_mean_ap
    },
    {
        "Model": "Random forest",
        "Real-label CV AP": real_forest_ap,
        "Shuffled-label CV AP": shuffled_forest_search.best_score_,
        "No-skill baseline AP": baseline_mean_ap
    }
])

comparison["Real minus shuffled"] = (
    comparison["Real-label CV AP"]
    - comparison["Shuffled-label CV AP"]
)

comparison["Shuffled gain over baseline"] = (
    comparison["Shuffled-label CV AP"]
    - comparison["No-skill baseline AP"]
)

display(comparison.round(5))

print("\nShuffled logistic best parameters:")
print(shuffled_logistic_search.best_params_)

print("\nShuffled forest best parameters:")
print(shuffled_forest_search.best_params_)

,Model,Real-label CV AP,Shuffled-label CV AP,No-skill baseline AP,Real minus shuffled,Shuffled gain over baseline
0,Logistic regression,0.0533,0.05694,0.05422,-0.00364,0.00271
1,Random forest,0.0562,0.05945,0.05422,-0.00325,0.00522



Shuffled logistic best parameters:
{'model__C': np.float64(2.6373339933815254), 'model__class_weight': None, 'pre__categorical__imputer__strategy': 'constant'}

Shuffled forest best parameters:
{'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': np.float64(0.0018559980846490584), 'model__n_estimators': 80, 'pre__categorical__imputer__strategy': 'most_frequent'}


## 13. Fold-by-fold permutation diagnostics


In [13]:
fold_comparison_rows = []

for fold in range(3):

    # Reconstruct the chronological validation labels
    _, validation_idx = list(
        shuffled_cv.split(X_train_check)
    )[fold]

    baseline_ap = shuffled_y.iloc[
        validation_idx
    ].mean()

    # Retrieve scores from the selected configurations
    real_scores = {
        "Logistic": [0.03959, 0.06715, 0.05311],
        "Forest": [0.04118, 0.07479, 0.05250]
    }

    for name, search in [
        ("Logistic", shuffled_logistic_search),
        ("Forest", shuffled_forest_search)
    ]:

        shuffled_ap = search.cv_results_[
            f"split{fold}_test_score"
        ][search.best_index_]

        real_ap = real_scores[name][fold]

        fold_comparison_rows.append({
            "Fold": fold + 1,
            "Model": name,
            "Baseline AP": baseline_ap,
            "Real AP": real_ap,
            "Shuffled AP": shuffled_ap,
            "Real gain": real_ap - baseline_ap,
            "Shuffled gain": shuffled_ap - baseline_ap
        })


shuffled_fold_comparison = pd.DataFrame(
    fold_comparison_rows
)

display(
    shuffled_fold_comparison.round(5)
)

,Fold,Model,Baseline AP,Real AP,Shuffled AP,Real gain,Shuffled gain
0,1,Logistic,0.04047,0.03959,0.03913,-0.00088,-0.00133
1,1,Forest,0.04047,0.04118,0.04710,0.00071,0.00664
2,2,Logistic,0.06394,0.06715,0.06556,0.00321,0.00162
3,2,Forest,0.06394,0.07479,0.06717,0.01085,0.00324
4,3,Logistic,0.05827,0.05311,0.06613,-0.00516,0.00785
5,3,Forest,0.05827,0.05250,0.06406,-0.00577,0.00579
